# L14 · FastAPI 入门：写出你的第一个 API

**学习目标**
- 理解为什么 FastAPI 是当今最流行的 Python API 框架
- 用极少量代码写出一个带「数据校验」的 API
- 在 notebook 内启动服务并用 requests 调用它

**前置依赖**：L13（HTTP/REST 概念）  
**预计时长**：45 分钟  
**技术栈**：`fastapi`、`uvicorn`、`requests`（需 pip 安装）

---

## 概念讲解：FastAPI = 写 API 的「傻瓜相机」

L13 我们手搓了服务器，但工业界用框架。`FastAPI` 好处：
- **极简**：一个函数 = 一个接口
- **自动校验**：你用「类型注解」声明参数，它自动检查格式
- **自带文档**：访问 `/docs` 就有可交互网页

它是 OpenAI、微软、Uber 都在用的现代框架。

## 第一步：定义 App 与第一个接口

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="我的第一个API")

class Item(BaseModel):
    name: str
    price: float

@app.post("/items")
def create_item(item: Item):
    return {"ok": True, "你提交的商品": item.name, "税后价格": round(item.price * 1.1, 2)}

## 第二步：在后台启动服务器

In [ ]:
import uvicorn, threading, time
PORT = 8770
def run():
    uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning")
threading.Thread(target=run, daemon=True).start()
time.sleep(2)   # 等服务器起来
print(f"✅ FastAPI 已在 http://127.0.0.1:{PORT} 运行，打开 /docs 看自动文档")

## 第三步：用 requests 当客户端调用

In [ ]:
import requests
r = requests.post(f"http://127.0.0.1:{PORT}/items",
                  json={"name": "机械键盘", "price": 399})
print(r.json())

# 🎯 AHA 顿悟单元格：你的 API 会自动「拒绝脏数据」

运行下面代码。你会看到：
1. 正常数据 → 正常返回；
2. **故意传错类型（把价格写成文字）** → FastAPI 自动返回清晰的错误，根本不会执行你的函数！

> 这就是「类型注解 = 免费的数据安检门」。工业级 AI 服务的稳定性，很大程度来自这种自动校验。
> 你写的第一个 API，已经具备生产级防御。

In [ ]:
# ===== 运行我！看正确与错误两种调用 =====
import requests, json
print("  ① 正确调用：")
ok = requests.post(f"http://127.0.0.1:{PORT}/items", json={"name": "咖啡", "price": 28.0})
print("   " + json.dumps(ok.json(), ensure_ascii=False))

print("\n  ② 故意传错（价格写成文字）：")
bad = requests.post(f"http://127.0.0.1:{PORT}/items", json={"name": "咖啡", "price": "很贵"})
print("   状态码：", bad.status_code)               # 422 = 校验失败
print("   错误信息：", bad.json()["detail"][0]["msg"])
print("\n  🛡️  看！坏数据在入口就被挡下，函数内部绝不会崩溃。")

# 📝 讲师备课笔记（接手 Agent 专用）

**本课难点**：`BaseModel`(pydantic) 数据模型；装饰器 `@app.post` 概念（先会用，L 后续展开）。  
**易错点**：服务器未启动就 requests 会连接失败（已加 sleep(2)）；端口冲突换 8770。  
**AHA 机制**：自动校验脏数据返回 422，直观展示「框架帮你守住入口」，强生产级质感。  
**依赖**：`pip install fastapi uvicorn pydantic requests`。  
**衔接**：L15 请求响应细节（路径参数/查询参数）；L16 数据库持久化。  
**备注**：`/docs` 在 notebook 内无法自动打开，需提示学员在浏览器访问（本机）。

# 📚 作业 / 下一步

1. 把 `Item` 加一个 `stock: int` 字段，再调用看校验。
2. 在浏览器打开 `http://127.0.0.1:8770/docs` 体验自动文档。
3. 下一课 **L15 请求与响应：前后端如何握手** —— 搞懂路径参数、查询参数、状态码。